In [1]:
"""
Data Cleaning & Standardization
Objective: To clean the data, fix missing values and make it industry standard.
Required Python Packages
- pandas
- Numpy

Tasks performed:
- Raw datasets Loaded
- Dataset compatibility assessment
- HDI dataset reshaping
- Standardize identifiers
- Validate joins
- Handle missing values
- Correct data types
- Export cleaned datasets

"""

'\nData Cleaning & Standardization\nObjective: To clean the data, fix missing values and make it industry standard.\nRequired Python Packages\n- pandas\n- Numpy\n\nTasks performed:\n- Raw datasets Loaded\n- Dataset compatibility assessment\n- HDI dataset reshaping\n- Standardize identifiers\n- Validate joins\n- Handle missing values\n- Correct data types\n- Export cleaned datasets\n\n'

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
data_path = Path("../data/raw")

World_Bank_File = data_path / "WorldBank.xlsx"
HDI_File = data_path / "HDI.csv"
Dictionary_File = data_path / "world_indicators_data_dictionary.csv"

In [4]:
world_bank = pd.read_excel(World_Bank_File)
HDI = pd.read_csv(HDI_File)
Dictionary = pd.read_csv(Dictionary_File, encoding = 'latin1')

In [5]:
print("World Bank:", world_bank.shape)
print("HDI:", HDI.shape)
print("Data Dictionary:", Dictionary.shape)

World Bank: (12449, 15)
HDI: (206, 1008)
Data Dictionary: (58, 3)


In [6]:
HDI.columns #here we are trying to confirm, if melt is required.

Index(['iso3', 'country', 'hdicode', 'region', 'hdi_rank_2021', 'hdi_1990',
       'hdi_1991', 'hdi_1992', 'hdi_1993', 'hdi_1994',
       ...
       'mf_2012', 'mf_2013', 'mf_2014', 'mf_2015', 'mf_2016', 'mf_2017',
       'mf_2018', 'mf_2019', 'mf_2020', 'mf_2021'],
      dtype='str', length=1008)

In [7]:
id_columns = ["iso3", "country", "region", "hdicode"]
hdi_long = pd.melt(frame = HDI, id_vars = id_columns, var_name = 'Indicator_Year', value_name = 'Value')

In [8]:
hdi_long.head()

,iso3,country,region,hdicode,Indicator_Year,Value
0,AFG,Afghanistan,SA,Low,hdi_rank_2021,180.0
1,AGO,Angola,SSA,Medium,hdi_rank_2021,148.0
2,ALB,Albania,ECA,High,hdi_rank_2021,67.0
3,AND,Andorra,NaN,Very High,hdi_rank_2021,40.0
4,ARE,United Arab Emirates,AS,Very High,hdi_rank_2021,26.0


In [9]:
hdi_long.shape

(206824, 6)

In [10]:
hdi_long["Indicator_Year"].sample(20)

15896           eys_2002
177945       lfpr_m_2010
139666          abr_2016
53018          le_f_2021
10430            le_2007
38852           gdi_2016
81664          le_m_2000
46107         hdi_f_2019
15606           eys_2000
185305         phdi_2013
109158    coef_ineq_2013
48434          le_f_1999
68615      gni_pc_f_2001
182854         phdi_2001
158195         pr_f_2010
94531         mys_m_1998
66783      gni_pc_f_1992
12335            le_2016
22704           mys_2003
144757         se_f_2009
Name: Indicator_Year, dtype: str

In [11]:
hdi_long["Indicator_Year"].unique()[:20]

<ArrowStringArray>
['hdi_rank_2021',      'hdi_1990',      'hdi_1991',      'hdi_1992',
      'hdi_1993',      'hdi_1994',      'hdi_1995',      'hdi_1996',
      'hdi_1997',      'hdi_1998',      'hdi_1999',      'hdi_2000',
      'hdi_2001',      'hdi_2002',      'hdi_2003',      'hdi_2004',
      'hdi_2005',      'hdi_2006',      'hdi_2007',      'hdi_2008']
Length: 20, dtype: str

In [12]:
hdi_long[["Indicator", "Year"]] = hdi_long["Indicator_Year"].str.extract(r"(.+)_(\d{4})$")

In [13]:
hdi_long.head()

,iso3,country,region,hdicode,Indicator_Year,Value,Indicator,Year
0,AFG,Afghanistan,SA,Low,hdi_rank_2021,180.0,hdi_rank,2021
1,AGO,Angola,SSA,Medium,hdi_rank_2021,148.0,hdi_rank,2021
2,ALB,Albania,ECA,High,hdi_rank_2021,67.0,hdi_rank,2021
3,AND,Andorra,NaN,Very High,hdi_rank_2021,40.0,hdi_rank,2021
4,ARE,United Arab Emirates,AS,Very High,hdi_rank_2021,26.0,hdi_rank,2021


In [14]:
hdi_long.drop(columns="Indicator_Year", inplace=True)

In [15]:
hdi_long["Year"] = hdi_long["Year"].astype("Int64")

In [16]:
hdi_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 206824 entries, 0 to 206823
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   iso3       206824 non-null  str    
 1   country    206824 non-null  str    
 2   region     151604 non-null  str    
 3   hdicode    191764 non-null  str    
 4   Value      183687 non-null  float64
 5   Indicator  206824 non-null  str    
 6   Year       206824 non-null  Int64  
dtypes: Int64(1), float64(1), str(5)
memory usage: 16.4 MB


In [17]:
hdi_long.head()

,iso3,country,region,hdicode,Value,Indicator,Year
0,AFG,Afghanistan,SA,Low,180.0,hdi_rank,2021
1,AGO,Angola,SSA,Medium,148.0,hdi_rank,2021
2,ALB,Albania,ECA,High,67.0,hdi_rank,2021
3,AND,Andorra,NaN,Very High,40.0,hdi_rank,2021
4,ARE,United Arab Emirates,AS,Very High,26.0,hdi_rank,2021


In [18]:
world_bank.head()

,Country Name,Country Code,Region,IncomeGroup,Year,"Birth rate, crude (per 1,000 people)","Death rate, crude (per 1,000 people)",Electric power consumption (kWh per capita),GDP (USD),GDP per capita (USD),Individuals using the Internet (% of population),"Infant mortality rate (per 1,000 live births)",Life expectancy at birth (years),Population density (people per sq. km of land area),Unemployment (% of total labor force) (modeled ILO estimate)
0,Afghanistan,AFG,South Asia,Low income,2018,NaN,NaN,NaN,1.936300e+10,520.897,NaN,47.9,NaN,56.9378,1.542
1,Afghanistan,AFG,South Asia,Low income,2017,33.211,6.575,NaN,2.019180e+10,556.302,13.50,49.5,64.130,55.5960,1.559
2,Afghanistan,AFG,South Asia,Low income,2016,33.981,6.742,NaN,1.936260e+10,547.228,11.20,51.2,63.763,54.1971,1.634
3,Afghanistan,AFG,South Asia,Low income,2015,34.809,6.929,NaN,1.990710e+10,578.466,8.26,53.1,63.377,52.7121,1.679
4,Afghanistan,AFG,South Asia,Low income,2014,35.706,7.141,NaN,2.048490e+10,613.856,7.00,55.1,62.966,51.1148,1.735


In [19]:
world_bank.info()

<class 'pandas.DataFrame'>
RangeIndex: 12449 entries, 0 to 12448
Data columns (total 15 columns):
 #   Column                                                        Non-Null Count  Dtype  
---  ------                                                        --------------  -----  
 0   Country Name                                                  12449 non-null  str    
 1   Country Code                                                  12449 non-null  str    
 2   Region                                                        12449 non-null  str    
 3   IncomeGroup                                                   12449 non-null  str    
 4   Year                                                          12449 non-null  int64  
 5   Birth rate, crude (per 1,000 people)                          11440 non-null  float64
 6   Death rate, crude (per 1,000 people)                          11416 non-null  float64
 7   Electric power consumption (kWh per capita)                   5848 non-null   

In [20]:
# Even after the melt, indicator-year column, still the grain of both the datasets is not matching, 
# world bank has one row for one country where as HDI has many rows for one country, due to multiple indicators.
# So we will again do wide -> long -> wide to pivot hdi, gdi, le, etc.
# This restructuring will help in joining the tables easily and prevent duplicity.

In [21]:
hdi_clean = hdi_long.pivot_table(index=["iso3", "country", "Year"], columns="Indicator", values="Value", aggfunc="first").reset_index()
# here we previously tried to pivot 'region' & ' hdiCode' as well but then,
# due to missing values pivot dropped many rows, which could a great difficulty in joining later

In [22]:
hdi_clean.head()

Indicator,iso3,country,Year,abr,co2_prod,coef_ineq,diff_hdi_phdi,eys,eys_f,eys_m,...,mmr,mys,mys_f,mys_m,phdi,pr_f,pr_m,rankdiff_hdi_phdi,se_f,se_m
0,AFG,Afghanistan,1990,142.960,0.209727,NaN,1.098901,2.50405,1.970663,3.939093,...,1500.0,0.971125,0.342503,1.311020,0.270,NaN,NaN,NaN,0.700485,5.419459
1,AFG,Afghanistan,1991,147.525,0.182525,NaN,1.075269,2.80655,2.096679,4.171640,...,1530.0,1.019356,0.371860,1.385486,0.276,NaN,NaN,NaN,0.772361,5.583395
2,AFG,Afghanistan,1992,147.521,0.095233,NaN,1.045296,3.10905,2.230753,4.417915,...,1520.0,1.067586,0.401218,1.459952,0.284,NaN,NaN,NaN,0.844236,5.747332
3,AFG,Afghanistan,1993,147.896,0.084285,NaN,1.010101,3.41155,2.373401,4.678730,...,1480.0,1.115817,0.430575,1.534418,0.294,NaN,NaN,NaN,0.916112,5.911269
4,AFG,Afghanistan,1994,155.669,0.075054,NaN,1.027397,3.71405,2.525171,4.954942,...,1540.0,1.164047,0.459933,1.608884,0.289,NaN,NaN,NaN,0.987988,6.075205


In [23]:
hdi_clean.shape

(6592, 42)

In [24]:
interim_path = Path("../data/interim")
hdi_long.to_csv(interim_path / "hdi_long.csv", index = False)
hdi_clean.to_csv(interim_path / "hdi_clean.csv", index = False)

In [25]:
wb_iso = set(world_bank["Country Code"].unique())
hdi_iso = set(hdi_clean["iso3"].unique())

print("World Bank ISO Codes :", len(wb_iso))
print("HDI ISO Codes        :", len(hdi_iso))

print("Common ISO Codes:", len(wb_iso & hdi_iso))

World Bank ISO Codes : 211
HDI ISO Codes        : 206
Common ISO Codes: 192


In [26]:
wb_only = sorted(wb_iso - hdi_iso)
print('Countries present in World Bank but missing from HDI:', wb_only)

hdi_only = sorted(hdi_iso - wb_iso)
print("\nCountries present in HDI but missing from World Bank:", hdi_only)

print("Common:", len(wb_iso & hdi_iso))

wb_only = sorted(wb_iso - hdi_iso)
hdi_only = sorted(hdi_iso - wb_iso)

Countries present in World Bank but missing from HDI: ['ABW', 'ASM', 'BMU', 'CHI', 'CUW', 'CYM', 'FRO', 'GRL', 'GUM', 'IMN', 'MAC', 'MAF', 'MNP', 'NCL', 'PRI', 'PYF', 'SXM', 'TCA', 'VIR']

Countries present in HDI but missing from World Bank: ['FSM', 'LAO', 'NRU', 'ZZA.VHHD', 'ZZB.HHD', 'ZZC.MHD', 'ZZD.LHD', 'ZZE.AS', 'ZZF.EAP', 'ZZG.ECA', 'ZZH.LAC', 'ZZI.SA', 'ZZJ.SSA', 'ZZK.WORLD']
Common: 192


In [27]:
HDI["iso3"].nunique()

206

In [28]:
hdi_long["iso3"].nunique()

206

In [29]:
hdi_clean["iso3"].nunique()

206

In [30]:
hdi_clean[["iso3","country"]].drop_duplicates().shape

(206, 2)

In [31]:
print(hdi_clean.shape)
print(hdi_clean["iso3"].nunique())
print(hdi_clean["country"].nunique())

(6592, 42)
206
206


In [32]:
hdi_clean[hdi_clean["iso3"].str.startswith("ZZ", na=False)][["iso3", "country"]].drop_duplicates()
# here we are trying to figure out whether to keep regional or global summaries or to remove them

Indicator,iso3,country
6240,ZZA.VHHD,Very high human development
6272,ZZB.HHD,High human development
6304,ZZC.MHD,Medium human development
6336,ZZD.LHD,Low human development
6368,ZZE.AS,Arab States
6400,ZZF.EAP,East Asia and the Pacific
6432,ZZG.ECA,Europe and Central Asia
6464,ZZH.LAC,Latin America and the Caribbean
6496,ZZI.SA,South Asia
6528,ZZJ.SSA,Sub-Saharan Africa


In [33]:
# Transformation
#Removed aggregate regional and development group records from the HDI dataset.

# Reason
# The project is conducted at the Country-Year level. Aggregate observations 
# such as "World", "South Asia", and "Very High Human Development" 
# do not represent individual countries 
# and would introduce inconsistencies during joins with the World Bank dataset.

# Validation
# Only ISO3 country codes representing individual countries were retained.

In [34]:
hdi_clean = hdi_clean[~hdi_clean["iso3"].str.startswith("ZZ", na=False)].copy()

In [35]:
print(hdi_clean.shape)
print(hdi_clean["iso3"].nunique())

(6240, 42)
195


In [36]:
wb_iso = set(world_bank["Country Code"].unique())
hdi_iso = set(hdi_clean["iso3"].unique())

wb_only = sorted(wb_iso - hdi_iso)
hdi_only = sorted(hdi_iso - wb_iso)

In [37]:
# Identifier Validation

# Objective:
# Verify that country identifiers are consistent between the World Bank and HDI datasets before integration.

# Validation:
# - Compared ISO3 country codes.
# - Identified unmatched country codes.
# - Investigated whether differences were caused by naming conventions or dataset coverage.

# Status:
# Investigation in progress.

In [38]:
print("Countries present in World Bank but missing from HDI:", wb_only)
print()
print("Countries present in HDI but missing from World Bank:", hdi_only)
print()
print("Common:", len(wb_iso & hdi_iso))

Countries present in World Bank but missing from HDI: ['ABW', 'ASM', 'BMU', 'CHI', 'CUW', 'CYM', 'FRO', 'GRL', 'GUM', 'IMN', 'MAC', 'MAF', 'MNP', 'NCL', 'PRI', 'PYF', 'SXM', 'TCA', 'VIR']

Countries present in HDI but missing from World Bank: ['FSM', 'LAO', 'NRU']

Common: 192


In [39]:
world_bank[world_bank["Country Name"].str.contains("Micronesia", case=False, na=False)]

,Country Name,Country Code,Region,IncomeGroup,Year,"Birth rate, crude (per 1,000 people)","Death rate, crude (per 1,000 people)",Electric power consumption (kWh per capita),GDP (USD),GDP per capita (USD),Individuals using the Internet (% of population),"Infant mortality rate (per 1,000 live births)",Life expectancy at birth (years),Population density (people per sq. km of land area),Unemployment (% of total labor force) (modeled ILO estimate)


In [40]:
world_bank[world_bank["Country Name"].str.contains("Lao", case=False, na=False)]

,Country Name,Country Code,Region,IncomeGroup,Year,"Birth rate, crude (per 1,000 people)","Death rate, crude (per 1,000 people)",Electric power consumption (kWh per capita),GDP (USD),GDP per capita (USD),Individuals using the Internet (% of population),"Infant mortality rate (per 1,000 live births)",Life expectancy at birth (years),Population density (people per sq. km of land area),Unemployment (% of total labor force) (modeled ILO estimate)


In [41]:
world_bank[world_bank["Country Name"].str.contains("Nauru", case=False, na=False)]

,Country Name,Country Code,Region,IncomeGroup,Year,"Birth rate, crude (per 1,000 people)","Death rate, crude (per 1,000 people)",Electric power consumption (kWh per capita),GDP (USD),GDP per capita (USD),Individuals using the Internet (% of population),"Infant mortality rate (per 1,000 live births)",Life expectancy at birth (years),Population density (people per sq. km of land area),Unemployment (% of total labor force) (modeled ILO estimate)


In [42]:
hdi_clean[hdi_clean["iso3"].isin(["FSM", "LAO", "NRU"])][["iso3", "country"]].drop_duplicates()

Indicator,iso3,country
1888,FSM,Micronesia (Federated States of)
3072,LAO,Lao People's Democratic Republic
4224,NRU,Nauru


In [43]:
sorted(world_bank["Country Name"].unique())

['Afghanistan',
 'Albania',
 'Algeria',
 'American Samoa',
 'Andorra',
 'Angola',
 'Antigua and Barbuda',
 'Argentina',
 'Armenia',
 'Aruba',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bahamas, The',
 'Bahrain',
 'Bangladesh',
 'Barbados',
 'Belarus',
 'Belgium',
 'Belize',
 'Benin',
 'Bermuda',
 'Bhutan',
 'Bolivia',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'Brunei Darussalam',
 'Bulgaria',
 'Burkina Faso',
 'Burundi',
 'Cabo Verde',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Cayman Islands',
 'Central African Republic',
 'Chad',
 'Channel Islands',
 'Chile',
 'China',
 'Colombia',
 'Comoros',
 'Congo, Dem. Rep.',
 'Congo, Rep.',
 'Costa Rica',
 "Cote d'Ivoire",
 'Croatia',
 'Cuba',
 'Curacao',
 'Cyprus',
 'Czech Republic',
 'Denmark',
 'Djibouti',
 'Dominica',
 'Dominican Republic',
 'Ecuador',
 'Egypt, Arab Rep.',
 'El Salvador',
 'Equatorial Guinea',
 'Eritrea',
 'Estonia',
 'Eswatini',
 'Ethiopia',
 'Faroe Islands',
 'Fiji',
 'Finland',
 'France',
 'French Polynesia',
 'Gab

In [44]:
world_bank[
    world_bank["Country Code"] == "LAO"
]

,Country Name,Country Code,Region,IncomeGroup,Year,"Birth rate, crude (per 1,000 people)","Death rate, crude (per 1,000 people)",Electric power consumption (kWh per capita),GDP (USD),GDP per capita (USD),Individuals using the Internet (% of population),"Infant mortality rate (per 1,000 live births)",Life expectancy at birth (years),Population density (people per sq. km of land area),Unemployment (% of total labor force) (modeled ILO estimate)


In [45]:
world_bank[
    world_bank["Country Name"].str.contains("Lao|Laos", case=False, na=False)
]

,Country Name,Country Code,Region,IncomeGroup,Year,"Birth rate, crude (per 1,000 people)","Death rate, crude (per 1,000 people)",Electric power consumption (kWh per capita),GDP (USD),GDP per capita (USD),Individuals using the Internet (% of population),"Infant mortality rate (per 1,000 live births)",Life expectancy at birth (years),Population density (people per sq. km of land area),Unemployment (% of total labor force) (modeled ILO estimate)


In [46]:
# Identifier Validation

# Comparison of ISO3 country codes between 
#the World Bank and HDI datasets identified a small number of unmatched records.

# Investigation confirmed that:
# - Aggregate HDI records (ISO codes beginning with `ZZ`) were removed because 
#   they represent regional or development-group summaries 
#   rather than individual countries.
# - A small number of countries (e.g., LAO, FSM, NRU) 
#   are present in the HDI dataset but absent from 
#   the curated World Bank dataset.
# - Several territories (e.g., Aruba, Bermuda, Guam) exist in 
#   the World Bank dataset but are not included in the HDI dataset.

# These differences are attributed to **dataset coverage** rather than 
# data quality issues. No manual mapping was performed.

In [47]:
def missing_value_summary(df):
    summary = df.isnull().sum().to_frame(name = "Missing Values")
    summary["Missing %"] = (summary["Missing Values"] / len(df) * 100).round(2)
    return summary.sort_values("Missing %", ascending = False)

In [48]:
missing_value_summary(world_bank)

,Missing Values,Missing %
Individuals using the Internet (% of population),7385,59.32
Unemployment (% of total labor force) (modeled ILO estimate),7241,58.17
Electric power consumption (kWh per capita),6601,53.02
GDP per capita (USD),2874,23.09
GDP (USD),2871,23.06
"Infant mortality rate (per 1,000 live births)",2465,19.80
Life expectancy at birth (years),1273,10.23
"Death rate, crude (per 1,000 people)",1033,8.30
"Birth rate, crude (per 1,000 people)",1009,8.11
Population density (people per sq. km of land area),604,4.85


In [49]:
missing_value_summary(hdi_clean)

,Missing Values,Missing %
Indicator,,
rankdiff_hdi_phdi,6085,97.52
gii_rank,6070,97.28
gdi_group,6068,97.24
hdi_rank,6049,96.94
ihdi,4504,72.18
loss,4504,72.18
coef_ineq,4504,72.18
ineq_inc,4418,70.80
ineq_edu,4261,68.29


In [50]:
world_bank.info()
hdi_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 12449 entries, 0 to 12448
Data columns (total 15 columns):
 #   Column                                                        Non-Null Count  Dtype  
---  ------                                                        --------------  -----  
 0   Country Name                                                  12449 non-null  str    
 1   Country Code                                                  12449 non-null  str    
 2   Region                                                        12449 non-null  str    
 3   IncomeGroup                                                   12449 non-null  str    
 4   Year                                                          12449 non-null  int64  
 5   Birth rate, crude (per 1,000 people)                          11440 non-null  float64
 6   Death rate, crude (per 1,000 people)                          11416 non-null  float64
 7   Electric power consumption (kWh per capita)                   5848 non-null   

In [51]:
processed_path = Path("../data/processed")

world_bank.to_csv(processed_path / "world_bank_clean.csv", index=False)
hdi_clean.to_csv(processed_path / "hdi_clean.csv", index=False)

In [52]:
print(world_bank.shape)
print(hdi_clean.shape)

(12449, 15)
(6240, 42)
